# CASMI 2026 - DreaMS Channel 5 (Colab track)

Builds an **independent fifth evidence channel** for the V17 pipeline by attaching a trained head
to the frozen DreaMS backbone. You cannot retrain FPNet (no training code), so this adds a new
channel rather than replacing one.

**Strategy, forced by the budget.** 100 Colab units is ~8h A100 / ~20h L4 - not enough to
fine-tune a transformer, but far more than enough to *embed the corpus once*. So: freeze the
backbone, cache 1024-d embeddings, and train heads on the cached vectors. Head training then
takes minutes on any hardware, so **you keep iterating after the Colab window closes.**

## What was verified against the DreaMS source before writing this

Read from `pluskal-lab/DreaMS@main` so you do not lose the window to a wrong guess:

| Fact | Consequence |
|---|---|
| `RawSpectraDataset.__getitem__` uses **only** `spectrum` + `precursor_mz` | No adduct/charge/collision-energy columns needed. Avoids writing string columns into HDF5, which h5py rejects for numpy `U` dtype. |
| Peak lists are **`(2, n_peaks)`** - row 0 m/z, row 1 intensity | `su.trim_peak_list` docstring; `high_form=False` is passed. Wrong orientation = silent garbage. |
| `MSData.from_pandas(df, hdf5_pth=...)` exists | No 2.5M-spectrum round-trip through `.mgf`. |
| `PreTrainedModel.from_name` accepts **only** `'DreaMS_embedding'` | `available_models()` also lists two names whose branches are commented out in source. |
| `normalize_mzs` defaults **False** and is never set by `dreams_predictions` | `DataFormatA.max_mz = 1000` is **never applied**. Test masses to 1,159 Da pass through un-clipped - out-of-distribution, but not broken. Flagged as a feature, not filtered. |
| Precursor peak is **prepended** at intensity `prec_intens=1.1` | DreaMS still sees the precursor for the ~44% of test spectra where it is absent from the peak list. |
| `n_highest_peaks = 100` for the embedding model | Our preprocessing caps at 100 to match. |

## Run order

Stages are checkpointed to Drive, so a dropped session costs one stage, not the day.

1. Setup + install (~10 min)
2. Verify paths + filter the corpus (~20 min) - no downloads, everything is already on Drive
3. **Embed** - the expensive, irreplaceable artifact. Do this first and back it up.
4. Fingerprint targets + scaffold split
5. Train head (minutes; iterate freely)
6. **Retrieval eval** against the real ~711k COCONUT+train pool - the number that decides
   whether this channel is worth shipping
7. Export + offline packaging for the Kaggle notebook

All inputs live under one folder, set by `CFG.ROOT` (default
`/content/drive/MyDrive/casmi_2026`). Stage 0 prints a manifest and fails fast, naming the
missing path, rather than erroring deep inside Stage 3.


In [ ]:
#@title Stage 0 - config
from pathlib import Path

class CFG:
    # --- one Drive folder holds everything; ROOT is read-only, WORK is ours ---
    ROOT        = "/content/drive/MyDrive/casmi_2026"
    WORK        = ROOT + "/_dreams_out"
    USE_DRIVE   = True

    # --- Stage 6 eval ---
    # True  = score against the real ~711k COCONUT+train pool the inference notebook uses,
    #         so the MRR@25 is directly comparable to your pipeline.
    # False = fall back to train-structure decoys (faster, but an optimistic proxy).
    USE_REAL_POOL = True
    EVAL_PPM      = 8.5

    # --- spectrum preprocessing (mirrors the plan's Tier-1 findings) ---
    # D6 MEASURED: raising the floor to 1% HURT Class-1 retrieval (0.9154 -> 0.9090).
    # The tour notebook's 1% advice is about comparing libraries fairly, not about
    # maximising retrieval. Keep the baseline's 0.2%.
    INT_FLOOR   = 0.002
    N_PEAKS     = 100      # must match PreTrainedModel n_highest_peaks for DreaMS_embedding
    DROP_ABOVE_PRECURSOR = 2.0   # Da; test set has these removed, riken/gnps still carry them

    # --- corpus selection ---
    TEST_ADDUCTS = ["[M+H]+","[M+NH4]+","[M-H2O+H]+","[M-2H2O+H]+","[M+Na]+","[M+K]+",
                    "[M-H]-","[M-H2O-H]-","[M+CH2O2-H]-","[M+Cl]-"]
    ONLY_TEST_ADDUCTS = True     # drops the 14.3% of train on dimer/exotic adducts
    # enveda-180 is 45% of spectra but ~0.004 Jaccard overlap with any NP library.
    # Keep some for timsTOF instrument adaptation, not all of it.
    CAP_ENVEDA180 = 150_000
    CAP_PER_LIB   = 400_000
    MAX_SPECTRA   = None         # e.g. 600_000 for a fast first pass; None = no cap

    # --- embedding ---
    CHUNK       = 200_000        # spectra per resumable chunk
    BATCH       = 256

    # --- head training ---
    HID         = 2048
    DROPOUT     = 0.15
    LR          = 3e-4
    EPOCHS      = 25
    BS          = 1024
    VAL_FRAC    = 0.12           # scaffold-disjoint

if CFG.USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

R = Path(CFG.ROOT)
COMP      = R / "enveda-CASMI26-molecule-id-mass-spectra"
TRAIN_P   = COMP / "train.parquet"
TEST_P    = COMP / "test.parquet"
COCO      = R / "coconut_dataset"
FPB       = COCO / "fp_bits.npy"
COCO_FP   = COCO / "coco_fp.npy"
COCO_MASS = COCO / "coco_mass.npy"
COCO_META = COCO / "coco_meta.pkl"
RANK_TRAIN = R / "casmi26-ranker-features" / "rank_train.npz"
FPNET_DIR  = R / "casmi26-fp-models-v2"
BIO_DIR    = R / "chebi-lipidmaps-casmi26"
RDKIT_WHL  = next(iter(sorted((R / "casmi26-offline-rdkit-2026033").glob("rdkit-*.whl"))), None)

W = Path(CFG.WORK)
W.mkdir(parents=True, exist_ok=True)
(W / "emb").mkdir(exist_ok=True)

REQUIRED = {"train.parquet": TRAIN_P, "fp_bits.npy": FPB}
OPTIONAL = {"coco_fp.npy": COCO_FP, "coco_mass.npy": COCO_MASS, "coco_meta.pkl": COCO_META,
            "rank_train.npz": RANK_TRAIN, "test.parquet": TEST_P,
            "rdkit wheel": RDKIT_WHL, "fpnet dir": FPNET_DIR, "bio dir": BIO_DIR}

def _size(p):
    if p is None or not p.exists(): return "-"
    if p.is_dir(): return f"{len(list(p.iterdir()))} files"
    n = p.stat().st_size
    return f"{n/1e9:.2f} GB" if n >= 1e9 else (f"{n/1e6:.1f} MB" if n >= 1e6 else f"{n/1e3:.0f} KB")

print(f"data root : {R}")
print(f"work dir  : {W}\n")
print(f"{'':3} {'file':16} {'size':>10}   path")
missing = []
for tag, group in (("REQ", REQUIRED), ("opt", OPTIONAL)):
    for name, p in group.items():
        ok = p is not None and p.exists()
        if not ok and tag == "REQ": missing.append(name)
        mark = "ok " if ok else ("!! " if tag == "REQ" else " . ")
        print(f"{mark} {name:16} {_size(p):>10}   {p}")
if missing:
    raise SystemExit(f"\nMISSING REQUIRED: {missing}\nCheck CFG.ROOT (currently {CFG.ROOT}).")
print("\nall required inputs present")

import torch
print("cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

---
## Stage 1 - Install DreaMS and fetch the backbone

`pip install -e .` from a clone. The weights (`embedding_model.ckpt`) download to
`dreams/models/pretrained/` on first use; we copy them to Drive so a session restart does not
re-download.


In [ ]:
#@title Stage 1 - install
import os, subprocess, sys, shutil
from pathlib import Path

if not Path("/content/DreaMS").exists():
    subprocess.run(["git","clone","--depth","1",
                    "https://github.com/pluskal-lab/DreaMS.git","/content/DreaMS"], check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","-e","/content/DreaMS"], check=True)

# rdkit is needed for the fingerprint targets. Prefer the pinned 2026.3.3 wheel from Drive so
# the fingerprints match what the competition's own RDKit produces; fall back to PyPI.
try:
    import rdkit
except ImportError:
    if RDKIT_WHL is not None and RDKIT_WHL.exists():
        r = subprocess.run([sys.executable,"-m","pip","install","-q",str(RDKIT_WHL)])
        if r.returncode != 0:
            subprocess.run([sys.executable,"-m","pip","install","-q","rdkit"], check=True)
    else:
        subprocess.run([sys.executable,"-m","pip","install","-q","rdkit"], check=True)
import rdkit; print("rdkit", rdkit.__version__)

import importlib
for m in ["dreams", "dreams.api", "dreams.utils.data", "dreams.utils.dformats"]:
    importlib.invalidate_caches()
from dreams.api import PreTrainedModel, dreams_embeddings
from dreams.definitions import DREAMS_EMBEDDING, PRETRAINED
import dreams.utils.data as du
import dreams.utils.dformats as dformats
print("DreaMS imported. model name:", DREAMS_EMBEDDING)

CKPT_CACHE = W / "embedding_model.ckpt"
PRETRAINED.mkdir(parents=True, exist_ok=True)
tgt = PRETRAINED / "embedding_model.ckpt"
if CKPT_CACHE.exists() and not tgt.exists():
    shutil.copy(CKPT_CACHE, tgt); print("restored ckpt from Drive")

model = PreTrainedModel.from_name(DREAMS_EMBEDDING)   # downloads on first run
print("n_highest_peaks:", model.n_highest_peaks)
assert model.n_highest_peaks == CFG.N_PEAKS, (
    f"CFG.N_PEAKS={CFG.N_PEAKS} must match the checkpoint's {model.n_highest_peaks}")

if tgt.exists() and not CKPT_CACHE.exists():
    shutil.copy(tgt, CKPT_CACHE); print("cached ckpt to Drive:", CKPT_CACHE)
print("backbone ready")

---
## Stage 2 - Load `fp_bits.npy`

Stage 0 already verified every path, so this cell only loads the bit index.

`fp_bits.npy` is the 6,930-bit selection the candidate pool is packed with. The head **must**
predict that exact layout: at inference its logits are dotted against `pool.fps()`, so a
different bit index is **silently wrong** rather than loudly broken. The sha1 of this array is
written into the exported checkpoint in Stage 7 so the Kaggle side can verify it matches.


In [ ]:
#@title Stage 2 - load fp_bits
import numpy as np

BITS  = np.load(FPB)
NBITS = len(BITS)
print("fp_bits:", BITS.shape, "dtype", BITS.dtype, "max index", int(BITS.max()))
# The concatenation is Morgan2(4096) | Morgan3(4096) | RDKitFP(2048) | MACCS(167) = 10,407 bits.
assert int(BITS.max()) < 10407, "fp_bits must index into the 10,407-bit concatenation"
assert len(np.unique(BITS)) == NBITS, "fp_bits contains duplicate indices"
print("NBITS =", NBITS)

---
## Stage 3 - Build the corpus and embed it

The expensive, irreplaceable step. Chunked and resumable: each chunk writes an `.npy` to Drive, so
a dropped session costs one chunk.

Preprocessing follows the **measured** Tier-0 results, not the priors: keep the 0.2% floor (the
1% floor measurably hurt retrieval), drop peaks above precursor + 2 Da, restrict to adducts that
can occur in test. DreaMS caps at 100 peaks anyway, which limits how much noise can get through.


In [ ]:
#@title Stage 3a - load, filter, build peak lists
import numpy as np, pandas as pd, pyarrow.parquet as pq, time

COLS = ["ingest_lib","inchikey14","normalized_smiles","molecular_formula",
        "ionization_mode","adduct","precursor_mz","ms2_mzs","ms2_normalized_intensities"]
t0=time.time()
tr = pq.read_table(TRAIN_P, columns=COLS).to_pandas()
print(f"loaded {len(tr):,} spectra in {time.time()-t0:.0f}s")

if CFG.ONLY_TEST_ADDUCTS:
    n0=len(tr); tr = tr[tr.adduct.isin(CFG.TEST_ADDUCTS)]
    print(f"adduct filter: {n0:,} -> {len(tr):,} ({(n0-len(tr))/n0*100:.1f}% dropped)")

rng = np.random.default_rng(0)
parts=[]
for lib, g in tr.groupby("ingest_lib"):
    cap = CFG.CAP_ENVEDA180 if lib=="enveda-180" else CFG.CAP_PER_LIB
    if cap and len(g) > cap:
        g = g.sample(cap, random_state=0)
    parts.append(g)
tr = pd.concat(parts).reset_index(drop=True)
if CFG.MAX_SPECTRA and len(tr) > CFG.MAX_SPECTRA:
    tr = tr.sample(CFG.MAX_SPECTRA, random_state=0).reset_index(drop=True)
print(f"corpus: {len(tr):,} spectra / {tr.inchikey14.nunique():,} structures")
print(tr.ingest_lib.value_counts().to_string())

def peak_list(mz, it, prec):
    """-> (2, n) float32, m/z ascending. None if the spectrum is unusable."""
    mz = np.asarray(mz, np.float64); it = np.asarray(it, np.float64)
    if len(mz) == 0: return None
    k = mz <= prec + CFG.DROP_ABOVE_PRECURSOR
    mz, it = mz[k], it[k]
    if len(mz) == 0: return None
    mx = it.max()
    if not np.isfinite(mx) or mx <= 0: return None
    k = it >= CFG.INT_FLOOR * mx
    mz, it = mz[k], it[k]
    if len(mz) < 2: return None
    if len(mz) > CFG.N_PEAKS:
        keep = np.sort(np.argsort(it)[-CFG.N_PEAKS:])
        mz, it = mz[keep], it[keep]
    o = np.argsort(mz)                      # m/z ascending
    mz, it = mz[o], it[o]
    return np.stack([mz, it / it.max()]).astype(np.float32)   # (2, n)

t0=time.time()
pls = [peak_list(a,b,p) for a,b,p in zip(tr.ms2_mzs, tr.ms2_normalized_intensities, tr.precursor_mz)]
ok = np.array([p is not None for p in pls])
tr = tr[ok].reset_index(drop=True)
pls = [p for p in pls if p is not None]
print(f"built {len(pls):,} peak lists in {time.time()-t0:.0f}s ({(~ok).sum():,} unusable dropped)")
assert pls[0].shape[0] == 2, "peak lists must be (2, n_peaks)"
print("example shape:", pls[0].shape, "| mz ascending:", bool(np.all(np.diff(pls[0][0])>=0)))

tr = tr.drop(columns=["ms2_mzs","ms2_normalized_intensities"])
tr["oob_mass"] = (tr.precursor_mz > 1000).astype(np.int8)   # DreaMS pretrained <=1000 Da
print(f"spectra above DreaMS's 1000 Da pretraining range: {tr.oob_mass.sum():,} "
      f"({tr.oob_mass.mean()*100:.2f}%) - kept, but flag this to the ranker")
tr.to_parquet(W/"corpus_meta.parquet")

In [ ]:
#@title Stage 3b - embed (resumable)
import numpy as np, pandas as pd, time, gc
from pathlib import Path
import dreams.utils.data as du

N = len(tr); nchunk = (N + CFG.CHUNK - 1)//CFG.CHUNK
print(f"{N:,} spectra -> {nchunk} chunks of {CFG.CHUNK:,}")

for ci in range(nchunk):
    out = W/"emb"/f"emb_{ci:04d}.npy"
    if out.exists():
        print(f"chunk {ci}: cached"); continue
    lo, hi = ci*CFG.CHUNK, min((ci+1)*CFG.CHUNK, N)
    t0 = time.time()
    df = pd.DataFrame({"spectrum": pls[lo:hi],
                       "precursor_mz": tr.precursor_mz.values[lo:hi].astype(np.float64)})
    h5 = Path(f"/content/chunk_{ci}.hdf5")
    msdata = du.MSData.from_pandas(df, n_highest_peaks=CFG.N_PEAKS, hdf5_pth=h5,
                                   in_mem=False, mode="r")
    embs = dreams_embeddings(msdata, batch_size=CFG.BATCH, progress_bar=True)
    embs = np.asarray(embs, dtype=np.float32)
    assert embs.shape == (hi-lo, 1024), f"unexpected embedding shape {embs.shape}"
    np.save(out, embs)
    print(f"chunk {ci}: {embs.shape} in {time.time()-t0:.0f}s -> {out.name}")
    del df, msdata, embs; h5.unlink(missing_ok=True); gc.collect()

E = np.concatenate([np.load(W/"emb"/f"emb_{i:04d}.npy") for i in range(nchunk)])
assert len(E) == N, f"{len(E)} embeddings for {N} spectra"
print("EMBEDDINGS:", E.shape, E.dtype, "| finite:", bool(np.isfinite(E).all()))
print("This is the artifact worth protecting. It is on Drive at", W/"emb")

---
## Stage 4 - Fingerprint targets and a scaffold-disjoint split

Targets must reproduce the candidate pool's layout **exactly**:
`[Morgan2(4096) | Morgan3(4096) | RDKitFP(2048) | MACCS(167)]` sliced by `fp_bits.npy`.
This is the same concatenation `fp_and_mass()` builds in the baseline notebook.

The split is on **Murcko scaffold**, not on rows and not even on `inchikey14`. A row split
scatters a `riken` compound's ~22 spectra across both sides; an `inchikey14` split still leaves
near-identical analogues on both sides. Only a scaffold split measures generalisation to new
chemistry, which is what Classes 2 and 3 are.


In [ ]:
#@title Stage 4a - fingerprints for unique structures
import numpy as np, time
from multiprocessing import Pool
from rdkit import Chem, RDLogger
from rdkit.Chem import rdFingerprintGenerator, MACCSkeys
from rdkit.Chem.Scaffolds import MurckoScaffold
RDLogger.DisableLog("rdApp.*")

_g = {}
def _init():
    _g["m2"] = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=4096)
    _g["m3"] = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=4096)
    _g["rk"] = rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=2048, maxPath=6)

def fp_of(smi):
    if not _g: _init()
    m = Chem.MolFromSmiles(smi)
    if m is None: return None
    try:
        v = np.concatenate([
            _g["m2"].GetFingerprintAsNumPy(m).astype(np.uint8),
            _g["m3"].GetFingerprintAsNumPy(m).astype(np.uint8),
            _g["rk"].GetFingerprintAsNumPy(m).astype(np.uint8),
            np.array(MACCSkeys.GenMACCSKeys(m), dtype=np.uint8)])
        assert len(v) == 10407
        return v[BITS]
    except Exception:
        return None

def scaf_of(smi):
    m = Chem.MolFromSmiles(smi)
    if m is None: return ""
    try: return MurckoScaffold.MurckoScaffoldSmiles(mol=m)
    except Exception: return ""

uni = tr.drop_duplicates("inchikey14")[["inchikey14","normalized_smiles"]].reset_index(drop=True)
print(f"{len(uni):,} unique structures")

t0=time.time()
with Pool(os.cpu_count()) as p:
    fps   = p.map(fp_of,   list(uni.normalized_smiles), chunksize=200)
    scafs = p.map(scaf_of, list(uni.normalized_smiles), chunksize=200)
print(f"fingerprints + scaffolds in {time.time()-t0:.0f}s")

good = np.array([f is not None for f in fps])
uni = uni[good].reset_index(drop=True)
FP  = np.stack([f for f in fps if f is not None]).astype(np.uint8)
uni["scaffold"] = [s for s,g in zip(scafs,good) if g]
print("FP matrix:", FP.shape, "| mean bits on:", FP.mean()*NBITS)
assert FP.shape[1] == NBITS

ik2row = {k:i for i,k in enumerate(uni.inchikey14)}
keep = tr.inchikey14.isin(ik2row).values
tr2, E2, = tr[keep].reset_index(drop=True), E[keep]
y_row = tr2.inchikey14.map(ik2row).values
print(f"usable spectra: {len(tr2):,}")

In [ ]:
#@title Stage 4b - scaffold-disjoint split
import numpy as np
rng = np.random.default_rng(7)
scafs = uni.scaffold.values
uniq_scaf = np.array(sorted(set(scafs)))
rng.shuffle(uniq_scaf)
n_val = int(len(uniq_scaf)*CFG.VAL_FRAC)
val_scaf = set(uniq_scaf[:n_val].tolist())

struct_is_val = np.array([s in val_scaf for s in scafs])
spec_is_val   = struct_is_val[y_row]
print(f"scaffolds  train {len(uniq_scaf)-n_val:,} / val {n_val:,}")
print(f"structures train {(~struct_is_val).sum():,} / val {struct_is_val.sum():,}")
print(f"spectra    train {(~spec_is_val).sum():,} / val {spec_is_val.sum():,}")

# leakage check: no scaffold may appear on both sides
tr_sc = set(scafs[~struct_is_val].tolist()); va_sc = set(scafs[struct_is_val].tolist())
assert not (tr_sc & va_sc), "scaffold leakage"
assert not (set(uni.inchikey14[~struct_is_val]) & set(uni.inchikey14[struct_is_val])), "structure leakage"
print("no scaffold or structure leakage across the split")

---
## Stage 5 - Train the fingerprint head

An MLP on frozen 1024-d embeddings. Minutes per run, so iterate freely - and you can keep doing
this after Colab expires, because the embeddings are cached.


In [ ]:
#@title Stage 5 - train
import numpy as np, torch, torch.nn as nn, time, math

dev = "cuda" if torch.cuda.is_available() else "cpu"
Xtr = torch.from_numpy(E2[~spec_is_val]); Xva = torch.from_numpy(E2[spec_is_val])
Ytr = torch.from_numpy(FP[y_row[~spec_is_val]]).float()
Yva = torch.from_numpy(FP[y_row[spec_is_val]]).float()
print("train", tuple(Xtr.shape), "val", tuple(Xva.shape))

mu, sd = Xtr.mean(0, keepdim=True), Xtr.std(0, keepdim=True).clamp(min=1e-6)

class Head(nn.Module):
    def __init__(s, d=1024, h=CFG.HID, o=NBITS, p=CFG.DROPOUT):
        super().__init__()
        s.net = nn.Sequential(
            nn.Linear(d,h), nn.GELU(), nn.Dropout(p),
            nn.Linear(h,h), nn.GELU(), nn.Dropout(p),
            nn.Linear(h,o))
    def forward(s,x): return s.net(x)

head = Head().to(dev)
opt = torch.optim.AdamW(head.parameters(), lr=CFG.LR, weight_decay=1e-2)
steps = CFG.EPOCHS * max(1, len(Xtr)//CFG.BS)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=CFG.LR, total_steps=steps, pct_start=0.1)
lossf = nn.BCEWithLogitsLoss()

def evaluate():
    head.eval(); tot, n = 0.0, 0
    with torch.no_grad():
        for i in range(0, len(Xva), 4096):
            xb = ((Xva[i:i+4096].to(dev)-mu.to(dev))/sd.to(dev))
            yb = Yva[i:i+4096].to(dev)
            tot += lossf(head(xb), yb).item()*len(xb); n += len(xb)
    head.train(); return tot/n

best, bad = math.inf, 0
t0=time.time()
for ep in range(CFG.EPOCHS):
    perm = torch.randperm(len(Xtr))
    run, nb = 0.0, 0
    for i in range(0, len(Xtr)-CFG.BS+1, CFG.BS):
        idx = perm[i:i+CFG.BS]
        xb = ((Xtr[idx].to(dev)-mu.to(dev))/sd.to(dev)); yb = Ytr[idx].to(dev)
        loss = lossf(head(xb), yb)
        opt.zero_grad(); loss.backward(); opt.step()
        if sched.last_epoch < steps-1: sched.step()
        run += loss.item(); nb += 1
    v = evaluate()
    flag = ""
    if v < best - 1e-5:
        best, bad = v, 0
        torch.save(head.state_dict(), W/"head_best.pt"); flag = " *"
    else:
        bad += 1
    print(f"ep {ep:02d}  train {run/max(nb,1):.5f}  val {v:.5f}{flag}")
    if bad >= 5: print("early stop"); break
head.load_state_dict(torch.load(W/"head_best.pt"))
print(f"best val BCE {best:.5f} in {time.time()-t0:.0f}s")

---
## Stage 6 - The number that actually matters

Validation BCE over 6,930 bits says almost nothing about whether the channel will rank the right
molecule first. This cell measures **retrieval MRR@25** on scaffold-disjoint molecules against
real mass-window decoys, and settles the corrected **Idea 3.2**.

**A correction worth understanding before reading the output.** It is tempting to say `f . z`
"only rewards on-bits" and that the full Bernoulli log-likelihood
`sum_k [f_k log p_k + (1-f_k) log(1-p_k)]` is better because it also penalises bits the model says
are absent. **That is false.** Because `z_k = log p_k - log(1-p_k)` exactly, the off-bit term
expands to `sum_k log(1-p_k)`, which does not depend on the candidate. The two scorers are
**rank-equivalent**. The `Bernoulli (control)` row below exists to demonstrate this empirically -
if it does not come out identical to `f . z`, something is wrong with the code, not the theory.

What genuinely changes the ranking:

| scorer | what it is |
|---|---|
| `f . z` | the baseline's dot product |
| `f . z / sqrt(on-bits)` | the baseline's size patch - a heuristic that breaks the probabilistic reading |
| `Bernoulli (control)` | must tie with `f . z`; a correctness check, not a candidate |
| `(f*w) . z` | **reliability weighting** - a bit the model predicts at chance should not vote |
| `f . (z - logit(prior))` | **prior correction** - a bit on in 99% of molecules carries almost no information; this is the pointwise-mutual-information form |
| `(f*w) . (z - logit(prior))` | both |

The last three are **pure post-processing and apply equally to the FPNet checkpoints you cannot
retrain** - which is what makes them unusually cheap score.


In [ ]:
#@title Stage 6 - retrieval eval + per-bit reliability
import numpy as np, torch

# predicted logits per validation MOLECULE (mean over its spectra, as inference does)
head.eval()
with torch.no_grad():
    Z = []
    for i in range(0, len(Xva), 4096):
        xb = ((Xva[i:i+4096].to(dev)-mu.to(dev))/sd.to(dev))
        Z.append(head(xb).float().cpu().numpy())
Z = np.concatenate(Z)

val_rows = y_row[spec_is_val]
mol_logits, mol_struct = {}, []
for r, z in zip(val_rows, Z):
    mol_logits.setdefault(r, []).append(z)
mol_struct = sorted(mol_logits)
ZM = np.stack([np.mean(mol_logits[r], 0) for r in mol_struct])
print(f"validation molecules: {len(mol_struct):,}")

# per-bit reliability = validation AUC, estimated cheaply via rank statistics
Yv = FP[np.array(mol_struct)].astype(np.float64)
P  = 1.0/(1.0+np.exp(-ZM))
auc = np.full(NBITS, 0.5)
for b in range(NBITS):
    y = Yv[:,b]; npos = y.sum()
    if npos < 5 or npos > len(y)-5: continue
    rk = np.argsort(np.argsort(P[:,b])) + 1
    auc[b] = (rk[y==1].sum() - npos*(npos+1)/2) / (npos*(len(y)-npos))
Wbit = np.clip((auc-0.5)*2.0, 0.0, 1.0)
print(f"per-bit AUC: median {np.median(auc):.3f} | bits with AUC>0.7: {(auc>0.7).sum():,}/{NBITS:,}")
print(f"reliability weight: mean {Wbit.mean():.3f}, zero for {(Wbit==0).sum():,} uninformative bits")

FPf = FP.astype(np.float32)
logp  = np.log(np.clip(P, 1e-6, 1-1e-6)); log1p_ = np.log(np.clip(1-P, 1e-6, 1-1e-6))

# per-bit prior over TRAIN structures only (never the val side)
prior  = FPf[~struct_is_val].mean(0).clip(1e-4, 1-1e-4)
zprior = np.log(prior/(1-prior)).astype(np.float32)
print(f"bit prior: median {np.median(prior):.4f} | bits on in >90% of molecules: {(prior>0.9).sum():,}")

# ---- exact masses of our structures (the query side) ----
from rdkit.Chem.Descriptors import ExactMolWt
from rdkit import Chem
mass = np.array([ExactMolWt(Chem.MolFromSmiles(s)) if Chem.MolFromSmiles(s) else np.nan
                 for s in uni.normalized_smiles])

# ---- candidate pool ----------------------------------------------------------------
# Real pool = COCONUT + the training structures COCONUT lacks, exactly as
# build_candidate_pool() assembles it in the inference notebook. Each pool row records
# where its fingerprint comes from: 0 = coco_fp.npy (bit-packed), 1 = our FP matrix.
import pickle
if CFG.USE_REAL_POOL:
    cmeta   = pickle.load(open(COCO_META, "rb"))
    co_keys = np.asarray(cmeta["keys"], dtype=object)
    co_mass = np.load(COCO_MASS).astype(np.float64)
    co_fp   = np.load(COCO_FP, mmap_mode="r")          # packed uint8, unpacked per window
    assert int(cmeta["nbits"]) == NBITS, (
        f"COCONUT pack is {cmeta['nbits']} bits but fp_bits.npy says {NBITS} - layouts differ")
    co_set  = set(co_keys.tolist())
    tr_only = np.array([k not in co_set for k in uni.inchikey14.values])
    keep_tr = tr_only & np.isfinite(mass)
    pool_mass = np.concatenate([co_mass, mass[keep_tr]])
    pool_key  = np.concatenate([co_keys, uni.inchikey14.values[keep_tr]])
    pool_src  = np.concatenate([np.zeros(len(co_mass), np.int8),
                                np.ones(int(keep_tr.sum()), np.int8)])
    pool_idx  = np.concatenate([np.arange(len(co_mass)), np.nonzero(keep_tr)[0]])
    print(f"real pool: {len(co_mass):,} COCONUT + {int(keep_tr.sum()):,} train-only "
          f"= {len(pool_mass):,} structures")
else:
    ok = np.isfinite(mass)
    pool_mass = mass[ok]; pool_key = uni.inchikey14.values[ok]
    pool_src  = np.ones(int(ok.sum()), np.int8); pool_idx = np.nonzero(ok)[0]
    co_fp = None
    print(f"proxy pool: {len(pool_mass):,} training structures only")

fin = np.isfinite(pool_mass)
pool_mass, pool_key = pool_mass[fin], pool_key[fin]
pool_src, pool_idx  = pool_src[fin], pool_idx[fin]
o = np.argsort(pool_mass)
pool_mass, pool_key = pool_mass[o], pool_key[o]
pool_src, pool_idx  = pool_src[o], pool_idx[o]
key2pool = {k: i for i, k in enumerate(pool_key)}

def pool_window(t, ppm=CFG.EVAL_PPM):
    a = np.searchsorted(pool_mass, t*(1-ppm/1e6), "left")
    b = np.searchsorted(pool_mass, t*(1+ppm/1e6), "right")
    return np.arange(a, b)

def pool_fps(sel):
    """(len(sel), NBITS) float32, assembled from whichever source each row came from."""
    out = np.zeros((len(sel), NBITS), np.float32)
    s, ix = pool_src[sel], pool_idx[sel]
    m0 = s == 0
    if m0.any():
        out[m0] = np.unpackbits(np.asarray(co_fp[ix[m0]]), axis=1)[:, :NBITS]
    if (~m0).any():
        out[~m0] = FPf[ix[~m0]]
    return out

NAMES = ["f.z", "f.z/sqrt(on)", "Bernoulli (control)",
         "(f*w).z", "f.(z-prior)", "(f*w).(z-prior)"]
scorers = {k: [] for k in NAMES}
ncand, n_missing = [], 0
for i, row in enumerate(mol_struct):
    if not np.isfinite(mass[row]): continue
    prow = key2pool.get(uni.inchikey14.values[row])
    if prow is None:            # true structure absent from the pool -> unscoreable, as at inference
        n_missing += 1; continue
    cand = pool_window(mass[row])
    if prow not in cand: cand = np.append(cand, prow)
    if len(cand) < 2: continue
    ncand.append(len(cand))
    f, z = pool_fps(cand), ZM[i]
    onb  = f.sum(1)
    raw  = f @ z
    vals = [raw,
            raw/np.sqrt(np.maximum(onb, 1.0)),
            f @ logp[i] + (1.0-f) @ log1p_[i],     # control: must tie with f.z
            f @ (Wbit*z),
            f @ (z - zprior),
            f @ (Wbit*(z - zprior))]
    tgt = int(np.where(cand == prow)[0][0])
    for name, s in zip(NAMES, vals):
        rank = int((s > s[tgt]).sum()) + 1
        scorers[name].append(1.0/rank if rank <= 25 else 0.0)
if n_missing:
    print(f"note: {n_missing} val structures are not in the pool at all "
          f"(they would score 0 at inference too; excluded here)")

print(f"\nmolecules scored: {len(ncand):,} | median candidates/window: {int(np.median(ncand))}")
print(f"{'scorer':26} {'MRR@25':>8} {'top-1':>8}")
res = {}
for k in NAMES:
    v = np.array(scorers[k]); res[k] = float(v.mean())
    print(f"{k:26} {v.mean():8.4f} {(v==1.0).mean()*100:7.1f}%")

ctrl = abs(res["Bernoulli (control)"] - res["f.z"])
print(f"\ncontrol check |Bernoulli - f.z| = {ctrl:.6f}  (must be ~0: they are rank-equivalent,")
print("  because z_k = log p_k - log(1-p_k) makes the off-bit term candidate-independent)")
if ctrl > 1e-6:
    print("  !! non-zero -> a bug in the scoring code, not a real improvement. Investigate.")

cands = {k: v for k, v in res.items() if k != "Bernoulli (control)"}
bestk = max(cands, key=cands.get)
print(f"\nVERDICT: best scorer is '{bestk}' at MRR@25 {res[bestk]:.4f} "
      f"({res[bestk]-res['f.z']:+.4f} vs the baseline's f.z)")
if res[bestk] > res["f.z"] + 0.005:
    print("         -> worth applying. It is post-processing over the logits, so use it on the")
    print("            EXISTING fp_*.pt output too - no retraining needed.")
else:
    print("         -> no material gain here. Do not spend more time on scorer variants.")
if CFG.USE_REAL_POOL:
    print("\nThese numbers come from the SAME ~711k COCONUT+train pool and the same 8.5 ppm window")
    print("the inference notebook uses, so they are comparable to your pipeline's Channel 4.")
    print("Remaining caveat: the queries are scaffold-disjoint but still Class-1-like (their")
    print("spectra existed in training), so Classes 2 and 3 are still unmeasured.")
else:
    print("\nCaveat: decoys are drawn from training structures only, so this is an optimistic proxy")
    print("        for the real COCONUT window. Use it to COMPARE scorers, not to predict LB.")

---
## Stage 7 - Export for the offline Kaggle notebook

Kaggle inference has **no internet**, so the backbone, the `dreams` package and its dependency
closure all have to ship as a Kaggle dataset. This cell assembles that folder.


In [ ]:
#@title Stage 7 - package
import shutil, subprocess, sys, json, hashlib, numpy as np, torch
from pathlib import Path

OUT = W/"kaggle_upload"; OUT.mkdir(exist_ok=True)

torch.save({
    "state_dict": head.state_dict(),
    "mu": mu.cpu(), "sd": sd.cpu(),
    "nbits": int(NBITS), "hid": CFG.HID, "dropout": CFG.DROPOUT, "in_dim": 1024,
    "bit_auc": auc.astype(np.float32),
    "bit_weight": Wbit.astype(np.float32),
    "bit_prior": prior.astype(np.float32),
    "bit_zprior": zprior.astype(np.float32),
    "best_scorer": bestk,
    "fp_bits_sha1": hashlib.sha1(BITS.tobytes()).hexdigest(),
    "n_peaks": CFG.N_PEAKS, "int_floor": CFG.INT_FLOOR,
    "drop_above_precursor": CFG.DROP_ABOVE_PRECURSOR,
    "val_mrr": {k: float(v) for k,v in res.items()},
}, OUT/"dreams_head.pt")
print("saved dreams_head.pt")

shutil.copy(PRETRAINED/"embedding_model.ckpt", OUT/"embedding_model.ckpt")
if (OUT/"DreaMS").exists(): shutil.rmtree(OUT/"DreaMS")
shutil.copytree("/content/DreaMS", OUT/"DreaMS",
                ignore=shutil.ignore_patterns(".git","data","experiments","*.ipynb"))

WH = OUT/"wheels"; WH.mkdir(exist_ok=True)
subprocess.run([sys.executable,"-m","pip","download","-q","-d",str(WH),"/content/DreaMS"],
               check=False)
print("wheels:", len(list(WH.glob('*'))))

tot = sum(f.stat().st_size for f in OUT.rglob("*") if f.is_file())
print(f"\nupload folder: {OUT}  ({tot/1e6:.0f} MB)")
print("""
Next:
  1. Download this folder and create a Kaggle dataset from it (or use
     `kaggle datasets create -p <folder>`).
  2. Attach it to the inference notebook alongside the existing inputs.
  3. Install offline in the first cell:
       !pip install -q --no-index --find-links=/kaggle/input/<ds>/wheels /kaggle/input/<ds>/DreaMS
     If that fails, `--no-deps` plus whatever Kaggle's image is already missing
     (pytorch-lightning is the usual gap).
""")

---
## Stage 8 - The Kaggle-side inference snippet

Paste into the baseline notebook. It produces `z_dreams`, a second logits vector exactly like
`model_logits(sub)`, so it slots straight into `rank_features`.

**Remember the blocker from the plan:** `X = rank_features(...)[:, :NFEAT]` truncates to the 31
columns of the attached `rank_train.npz`. Appending DreaMS features does nothing until you rebuild
that file. Until then, use this channel in one of two ways that need no rebuild:

- **average it into the existing `zlog`** (both predict the same 6,930 bits, so they are directly
  averageable - a free ensemble that flows through the existing `mfeat` block), or
- **swap the scorer** to whichever variant won Stage 6, applied to the combined logits.


In [ ]:
#@title Stage 8 - snippet to paste into the Kaggle inference notebook (not run here)
SNIPPET = r"""
# ---- DreaMS Channel 5 -------------------------------------------------------
import numpy as np, torch, torch.nn as nn, glob, os
from pathlib import Path
import dreams.utils.data as du
from dreams.api import dreams_embeddings, PreTrainedModel
from dreams.definitions import DREAMS_EMBEDDING, PRETRAINED

DS = os.path.dirname(find_file('dreams_head.pt'))
PRETRAINED.mkdir(parents=True, exist_ok=True)
import shutil; shutil.copy(os.path.join(DS,'embedding_model.ckpt'), PRETRAINED/'embedding_model.ckpt')
_ck = torch.load(os.path.join(DS,'dreams_head.pt'), map_location='cpu', weights_only=False)

class _Head(nn.Module):
    def __init__(s, d, h, o, p):
        super().__init__()
        s.net = nn.Sequential(nn.Linear(d,h), nn.GELU(), nn.Dropout(p),
                              nn.Linear(h,h), nn.GELU(), nn.Dropout(p), nn.Linear(h,o))
    def forward(s,x): return s.net(x)

_DEV  = 'cuda' if torch.cuda.is_available() else 'cpu'
_HEAD = _Head(_ck['in_dim'], _ck['hid'], _ck['nbits'], 0.0).to(_DEV).eval()
_HEAD.load_state_dict(_ck['state_dict'])
_MU, _SD = _ck['mu'].to(_DEV), _ck['sd'].to(_DEV)
_WBIT   = _ck['bit_weight']
_ZPRIOR = _ck['bit_zprior']
print('DreaMS head loaded | best scorer from Colab eval:', _ck.get('best_scorer'))

def _dreams_peaks(mz, it, prec, floor=_ck['int_floor'], n=_ck['n_peaks'],
                  above=_ck['drop_above_precursor']):
    mz = np.asarray(mz, np.float64); it = np.asarray(it, np.float64)
    if len(mz) == 0: return None
    k = mz <= prec + above; mz, it = mz[k], it[k]
    if len(mz) == 0: return None
    mx = it.max()
    if not np.isfinite(mx) or mx <= 0: return None
    k = it >= floor*mx; mz, it = mz[k], it[k]
    if len(mz) < 2: return None
    if len(mz) > n:
        keep = np.sort(np.argsort(it)[-n:]); mz, it = mz[keep], it[keep]
    o = np.argsort(mz); mz, it = mz[o], it[o]
    return np.stack([mz, it/it.max()]).astype(np.float32)   # (2, n)

@torch.no_grad()
def dreams_logits(sub, _tmp=Path('/kaggle/working/_dreams_tmp.hdf5')):
    rows, pls = [], []
    for r in sub.itertuples():
        p = _dreams_peaks(r.ms2_mzs, r.ms2_normalized_intensities, float(r.precursor_mz))
        if p is not None: pls.append(p); rows.append(float(r.precursor_mz))
    if not pls: return None
    import pandas as pd
    df = pd.DataFrame({'spectrum': pls, 'precursor_mz': np.array(rows, np.float64)})
    _tmp.unlink(missing_ok=True)
    msd = du.MSData.from_pandas(df, n_highest_peaks=_ck['n_peaks'], hdf5_pth=_tmp,
                                in_mem=True, mode='r')
    emb = np.asarray(dreams_embeddings(msd, batch_size=64, progress_bar=False), np.float32)
    x = (torch.from_numpy(emb).to(_DEV) - _MU) / _SD
    return _HEAD(x).float().mean(0).cpu().numpy()      # (nbits,)

# --- use it: average with the existing FPNet logits (needs no rank_train.npz rebuild) ---
#   zlog = model_logits(sub)
#   zdre = dreams_logits(sub)
#   if zdre is not None:
#       zlog = zdre if zlog is None else 0.5*(zlog + zdre)
# -----------------------------------------------------------------------------
"""
print(SNIPPET)
Path(W/"kaggle_snippet.py").write_text(SNIPPET)
print("saved to", W/"kaggle_snippet.py")

---
## If DreaMS fights you - fall back fast

The window is two days. **Hard stop the install at ~3 hours.** Fallbacks in order of closeness to
this task:

1. **MS2DeepScore** - pretrained on GNPS, predicts structural similarity *between two spectra*.
   Different shape (pairwise, not spectrum-to-fingerprint), but it would directly strengthen
   Channel 2's analogue retrieval, which is where Class 2 lives.
2. **MIST** - supervised spectrum-to-fingerprint, the closest architecture to what you need.
3. **Train a small transformer from scratch** on the same filtered corpus. You have the data and
   the targets from Stage 4; only the pretrained prior is lost.

Whichever you pick, **Stages 4-7 stay unchanged** - they only need an `(N, d)` embedding matrix.
That is the reason for the frozen-backbone design: the expensive part is replaceable.
